# SUBHAM — EfficientNet-B0 Training
### Team CodeCrafters (AI ML-21) — IEEE EMBS Pune 2026

**Run cells 1 to 10 in order. Do not skip any.**

In [ ]:
# CELL 1 — Install libraries
!pip install torchvision scikit-learn matplotlib seaborn -q
print('CELL 1 DONE: Libraries installed')

In [ ]:
# CELL 2 — Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2, json, os, glob
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'CELL 2 DONE: Using device = {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# CELL 3 — Upload Swarnali's 5 files
from google.colab import files
print('Upload: train.csv, val.csv, test.csv, class_weights.npy, label_map.json')
uploaded = files.upload()
print('CELL 3 DONE:', list(uploaded.keys()))

In [ ]:
# CELL 4 — Download HAM10000 and fix image paths
from google.colab import files as colab_files

# Upload kaggle.json
print('Upload your kaggle.json file...')
uploaded_kaggle = colab_files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle key configured. Downloading dataset...')

!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/ham10000 --unzip -q
print('Dataset downloaded!')

# Load CSV files
train_df = pd.read_csv('train.csv')
val_df   = pd.read_csv('val.csv')
test_df  = pd.read_csv('test.csv')
class_weights = np.load('class_weights.npy')
with open('label_map.json') as f:
    label_map = json.load(f)
idx_to_class = {v: k for k, v in label_map.items()}

# Fix image paths
image_paths = glob.glob('/content/ham10000/**/*.jpg', recursive=True)
image_dict = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
print(f'Images found: {len(image_dict)}')

train_df['image_path'] = train_df['image_id'].map(image_dict)
val_df['image_path']   = val_df['image_id'].map(image_dict)
test_df['image_path']  = test_df['image_id'].map(image_dict)

train_df = train_df.dropna(subset=['image_path']).reset_index(drop=True)
val_df   = val_df.dropna(subset=['image_path']).reset_index(drop=True)
test_df  = test_df.dropna(subset=['image_path']).reset_index(drop=True)

print(f'CELL 4 DONE: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}')

In [ ]:
# CELL 5 — Dataset class and DataLoaders
class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image_path'])
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
        if self.transform:
            img = self.transform(img)
        return img, int(row['label'])

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

train_dataset = SkinDataset(train_df, transform=train_transform)
val_dataset   = SkinDataset(val_df,   transform=val_transform)
test_dataset  = SkinDataset(test_df,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'CELL 5 DONE: Train batches={len(train_loader)} | Val batches={len(val_loader)}')

In [ ]:
# CELL 6 — Load EfficientNet-B0
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 7)
model = model.to(device)
print(f'CELL 6 DONE: EfficientNet-B0 loaded. Final layer = Linear({num_features}, 7)')

In [ ]:
# CELL 7 — Loss, Optimizer, Scheduler
weights_tensor = torch.FloatTensor(class_weights).to(device)
criterion  = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer  = optim.Adam(model.parameters(), lr=1e-4)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
print('CELL 7 DONE: Loss=WeightedCrossEntropy | Optimizer=Adam | Scheduler=ReduceLROnPlateau')

In [ ]:
# CELL 8 — TRAINING (wait 20-25 mins)
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        if (batch_idx + 1) % 50 == 0:
            print(f'   Batch {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f}')
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    accuracy = correct / total
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return total_loss / len(loader), accuracy, f1

NUM_EPOCHS = 20
best_val_acc = 0
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'val_f1':[]}

print(f'CELL 8 STARTED: Training for {NUM_EPOCHS} epochs...')
print('='*60)

for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}')
    print('-'*40)
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%')
    print(f'  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}% | Val F1: {val_f1:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/best_model.pth')
        print(f'  *** Best model saved! Val Acc: {best_val_acc*100:.2f}% ***')

print('\n' + '='*60)
print(f'CELL 8 DONE: Training complete! Best Val Accuracy = {best_val_acc*100:.2f}%')

In [ ]:
# CELL 9 — Plot training curves
epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train', markersize=4)
axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val',   markersize=4)
axes[0].set_title('Loss Curve', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-o', label='Train', markersize=4)
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   'r-o', label='Val',   markersize=4)
axes[1].set_title('Accuracy Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[2].plot(epochs, history['val_f1'], 'g-o', label='Val F1', markersize=4)
axes[2].set_title('Weighted F1 Score', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('F1 Score')
axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.suptitle('EfficientNet-B0 — CodeCrafters Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('CELL 9 DONE: Training curves saved')

In [ ]:
# CELL 10 — Final test evaluation + confusion matrix + save results
model.load_state_dict(torch.load('/content/best_model.pth'))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

test_acc = accuracy_score(all_labels, all_preds)
test_f1  = f1_score(all_labels, all_preds, average='weighted')
test_f1m = f1_score(all_labels, all_preds, average='macro')

print('='*55)
print('  FINAL RESULTS — SHARE THESE WITH CLAUDE')
print('='*55)
print(f'  Accuracy    : {test_acc*100:.2f}%')
print(f'  Weighted F1 : {test_f1:.4f}')
print(f'  Macro F1    : {test_f1m:.4f}')
print('='*55)

class_labels = ['nv','mel','bkl','bcc','akiec','vasc','df']
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=class_labels))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title(f'Confusion Matrix\nAccuracy: {test_acc*100:.2f}% | Weighted F1: {test_f1:.4f}', fontsize=13, fontweight='bold')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Save results
results = {'model':'EfficientNet-B0 (CodeCrafters)', 'dataset':'HAM10000',
           'accuracy': round(test_acc*100, 2), 'f1_weighted': round(test_f1, 4), 'f1_macro': round(test_f1m, 4)}
with open('/content/our_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('\nCELL 10 DONE: Download these files:')
print('  best_model.pth, training_curves.png, confusion_matrix.png, our_results.json')